# circscale: full sweep on Colab GPU

Runs the whole scaling-law experiment: LR tuning, then the main width sweep
(32 → 512 at MLP depth 4, constant-LR schedule, every eval checkpoint an
(N, D) datapoint). Roughly 30–60 min on a T4; the small widths are
dispatch-bound, the big ones are where the GPU pays off.

**Setup**: Runtime → Change runtime type → GPU. Every stage is idempotent —
completed runs are skipped and interrupted runs resume from their checkpoint
(≤2500 steps lost), so disconnects are cheap, and with `USE_DRIVE = True`
below, `runs/` lives in your Google Drive and survives the runtime entirely.

In [ ]:
# GPU + deps (Colab ships JAX; make sure it's the CUDA build)
!nvidia-smi -L
%pip -q install -U "jax[cuda12]" optax
import jax
print("jax", jax.__version__, jax.devices())
if jax.default_backend() == "cpu":
    print("WARNING: CPU backend — enable a GPU runtime and restart")

In [ ]:
import os

REPO_URL = "https://github.com/amdson/circscale.git"

if not os.path.exists("/content/circscale"):
    !git clone $REPO_URL /content/circscale
%cd /content/circscale
!git pull

In [ ]:
# Persist runs/ to Drive so nothing is lost when the runtime dies.
USE_DRIVE = True

import os
if USE_DRIVE and not os.path.islink("runs"):
    from google.colab import drive
    drive.mount("/content/drive")
    target = "/content/drive/MyDrive/circscale_runs"
    os.makedirs(target, exist_ok=True)
    assert not os.path.exists("runs"), "runs/ already exists as a plain dir"
    os.symlink(target, "runs")
print("runs ->", os.path.realpath("runs"))

In [ ]:
# Optional ~1 min sanity check of circuit + model on this backend.
!python -m pytest -q -x test_random_circuit.py test_mlp.py

## Stage 1: LR tuning

5k-step runs over LRs {3e-4, 1e-3, 3e-3} per width; writes
`runs/lr_table.json` and warns if any best LR sits on the grid edge (extend
`LR_GRID` in `sweep.py` and re-run this cell before trusting the sweep).

In [ ]:
from sweep import stage_lr_tune

stage_lr_tune()

## Stage 2: main sweep

Widths {32, 64, 128, 256, 512} at tuned LRs, 50k steps, seeds per width.
Interrupt any time; re-running the cell resumes.

In [ ]:
from sweep import stage_main

stage_main()

In [ ]:
!python sweep.py status

## Quick look

Sanity plots straight off the run files (the real fitting — envelope,
L(N, D) joint fit, bootstrap CIs — belongs in a separate analysis notebook).

In [ ]:
import glob

import matplotlib.pyplot as plt
import numpy as np

from train import load_run

runs = [load_run(p) for p in sorted(glob.glob("runs/*.npz"))]
print(f"{len(runs)} runs")


def n_params(c):
    w, d, r = c["width"], c["mlp_depth"], c["hidden_ratio"]
    return c["n_wires"] * 2 * w + d * (2 * r * w * w + w) + w

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# eval loss vs samples consumed, one curve per width (seed 0)
for c, d in runs:
    if c["model_seed"] != 0:
        continue
    D = d["eval_steps"][1:] * c["batch"]
    axes[0].plot(D, d["per_out_loss"][1:].mean(axis=1), label=f"w{c['width']}")
axes[0].set(xscale="log", yscale="log", xlabel="samples D", ylabel="eval BCE")
axes[0].axhline(np.log(2), color="gray", ls=":", lw=1)
axes[0].legend(fontsize=8); axes[0].set_title("L(D) per model size")

# final loss vs params, all seeds
for c, d in runs:
    axes[1].plot(n_params(c), d["per_out_loss"][-1].mean(), "o", color="tab:blue", alpha=0.6)
axes[1].set(xscale="log", yscale="log", xlabel="params N", ylabel="final eval BCE")
axes[1].set_title("L(N) at 50k steps")

# final accuracy vs output tap depth, per width (seed 0)
for c, d in runs:
    if c["model_seed"] != 0:
        continue
    depths, acc = d["out_depths"], d["per_out_acc"][-1]
    bins = np.arange(1, depths.max() + 2, 2)
    mids, means = [], []
    for lo in bins[:-1]:
        sel = (depths >= lo) & (depths < lo + 2)
        if sel.any():
            mids.append(lo + 0.5); means.append(acc[sel].mean())
    axes[2].plot(mids, means, label=f"w{c['width']}")
axes[2].axhline(0.5, color="gray", ls=":", lw=1)
axes[2].set(xlabel="output tap depth", ylabel="final eval accuracy")
axes[2].legend(fontsize=8); axes[2].set_title("hardness ladder")
plt.tight_layout()

In [ ]:
# If not using Drive: zip results and download before the runtime dies.
if not USE_DRIVE:
    !zip -qr circscale_runs.zip runs
    from google.colab import files
    files.download("circscale_runs.zip")